In [ ]:
%%capture
!pip install langchain-experimental langchain_community langgraph-checkpoint-sqlite tiktoken langchain-openai langchainhub chromadb langchain langgraph tavily-python

In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from google.colab import userdata

llm = AzureChatOpenAI(
    model=userdata.get('AZURE_MODEL_NAME'),
    deployment_name=userdata.get('AZURE_MODEL_NAME'),
    temperature=0,
    # max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=userdata.get('AZURE_API_KEY'),  # if you prefer to pass api key in directly instaed of using env vars
    azure_endpoint=userdata.get('AZURE_BASE_URL'),
    api_version=userdata.get('AZURE_API_VERSION'),
)

embd = AzureOpenAIEmbeddings(
    model=userdata.get('AZURE_EMBEDDING_NAME'),
    api_key=userdata.get('AZURE_API_KEY'),
    azure_endpoint=userdata.get('AZURE_BASE_URL'),
    api_version=userdata.get('AZURE_API_VERSION'),
)

In [ ]:
%pip install --upgrade --quiet  playwright > /dev/null
%pip install --upgrade --quiet  lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 27.6 MB/s eta 0:00:00


In [ ]:
!playwright install

164 MiB [] 0% 0.0s164 MiB [] 0% 28.1s164 MiB [] 0% 17.0s164 MiB [] 0% 8.8s164 MiB [] 1% 6.0s164 MiB [] 1% 5.2s164 MiB [] 2% 4.4s164 MiB [] 2% 4.2s164 MiB [] 3% 4.0s164 MiB [] 3% 3.6s164 MiB [] 4% 3.6s164 MiB [] 5% 3.4s164 MiB [] 6% 3.3s164 MiB [] 6% 3.1s164 MiB [] 7% 3.1s164 MiB [] 8% 2.9s164 MiB [] 8% 2.8s164 MiB [] 9% 2.8s164 MiB [] 10% 2.7s164 MiB [] 10% 2.6s164 MiB [] 11% 2.6s164 MiB [] 12% 2.5s164 MiB [] 13% 2.5s164 MiB [] 14% 2.5s164 MiB [] 14% 2.4s164 MiB [] 15% 2.4s164 MiB [] 16% 2.3s164 MiB [] 17% 2.2s164 MiB [] 18% 2.2s164 MiB [] 19% 2.1s164 MiB [] 20% 2.1s164 MiB [] 21% 2.1s164 MiB [] 22% 2.0s164 MiB [] 23% 2.0s164 MiB [] 24% 2.0s164 MiB [] 25% 2.0s164 MiB [] 26% 1.9s164 MiB [] 27% 1.9s164 MiB [] 28% 1.9s164 MiB [] 29% 1.8s164 MiB [] 30% 1.8s164 MiB [] 31% 1.8s164 MiB [] 32% 1.8s164 MiB [] 33% 1.8s164 MiB [] 33% 1.7s164 MiB [] 34% 1.7s164 MiB [] 35% 1.7s164 MiB [] 36% 1.7s164 MiB [] 37% 1.7s164 MiB [] 37% 1.6s164 MiB [] 39% 1.6s164 MiB [] 40% 1.5s164 MiB [] 41% 1.4s164 MiB [

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from langchain_community.document_loaders import AsyncChromiumLoader
from langchain_community.document_transformers import BeautifulSoupTransformer

# Load HTML
loader = AsyncChromiumLoader([
    "https://www.datacamp.com/blog/ai-chips",
    "https://www.datacamp.com/blog/ai-governance",
    "https://www.datacamp.com/blog/ai-ethics-introduction",
    # "https://www.congress.gov/bill/118th-congress/house-bill/6363/text?s=1&r=1",
    # "https://www.congress.gov/118/plaws/publ22/PLAW-118publ22.pdf",
    ])
html = loader.load()
# Transform
bs_transformer = BeautifulSoupTransformer()
docs_list = bs_transformer.transform_documents(html, tags_to_extract=["span"])
len(docs_list)

3

In [ ]:
# docs_list[:1]

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

### from langchain_cohere import CohereEmbeddings

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)
print(len(doc_splits))
# Add to vectorstore
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=embd,
)
retriever = vectorstore.as_retriever(search_kwargs=dict(k=22))

22


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        """You are a helpful assistant analyze some articles regarding AI breaking news topics using the following context to answer human question
        Context: {context}\n
    """),
    HumanMessagePromptTemplate.from_template("{question}")
])

In [ ]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from IPython.display import display, Markdown

# Prompt
# prompt = hub.pull("rlm/rag-prompt")


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.invoke("Generate a ppt a extensed presentation(10 ppts with detailed(5 bullet points, and then one extensed explanation for my speech)) about AI chips, AI governance and AI ethics")
display(Markdown(result))

Sure, here's a detailed outline for a 10-slide PowerPoint presentation on AI chips, AI governance, and AI ethics. Each slide will include 5 bullet points and an extended explanation for your speech.

---

### Slide 1: Title Slide
- **Title:** The Future of AI: Chips, Governance, and Ethics
- **Subtitle:** Understanding the Core Components of AI Technology
- **Your Name**
- **Date**
- **Institution/Organization**

**Speech:**
"Good [morning/afternoon/evening], everyone. Today, we will delve into the fascinating world of AI, focusing on three critical aspects: AI chips, AI governance, and AI ethics. These components are pivotal in shaping the future of artificial intelligence, ensuring it is both powerful and responsible."

---

### Slide 2: Introduction to AI Chips
- **Definition:** Specialized processors for AI tasks
- **Importance:** Accelerate AI computations
- **Types:** GPUs, TPUs, NPUs
- **Applications:** From smartphones to healthcare
- **Future Trends:** Increasing efficiency and capabilities

**Speech:**
"AI chips are the specialized processors designed to handle the complex computations required for AI tasks. They are crucial for accelerating AI processes, making them faster and more efficient. There are various types of AI chips, including GPUs, TPUs, and NPUs, each serving different purposes. These chips are used in a wide range of applications, from enhancing smartphone capabilities to revolutionizing healthcare. As we move forward, we can expect AI chips to become even more efficient and capable."

---

### Slide 3: Key Features of AI Chips
- **Specialized Processing Units:** Optimized for AI tasks
- **Memory Hierarchy:** Efficient data access
- **Energy Efficiency:** Reduced power consumption
- **Software Optimization:** Enhanced performance
- **Scalability:** Adaptable to various applications

**Speech:**
"AI chips come with several key features that set them apart from traditional processors. They have specialized processing units optimized for AI tasks, a memory hierarchy that ensures efficient data access, and are designed for energy efficiency to reduce power consumption. Software optimization plays a crucial role in enhancing their performance, and their scalability makes them adaptable to various applications."

---

### Slide 4: Applications of AI Chips
- **Robotics:** Industrial automation and healthcare robots
- **Smartphones:** Augmented reality and voice assistants
- **Healthcare:** Medical diagnostics and treatment plans
- **Finance:** Risk assessment and fraud detection
- **Smart Cities:** Environmental monitoring and smart homes

**Speech:**
"AI chips are integral to numerous applications. In robotics, they enable industrial automation and assistive healthcare robots. In smartphones, they power features like augmented reality and voice assistants. In healthcare, they facilitate medical diagnostics and personalized treatment plans. In finance, they enhance risk assessment and fraud detection. Additionally, AI chips are crucial for smart cities, enabling environmental monitoring and smart home technologies."

---

### Slide 5: Introduction to AI Governance
- **Definition:** Framework for responsible AI development
- **Importance:** Ensures ethical AI use
- **Components:** Laws, regulations, guidelines
- **Stakeholders:** Governments, academia, industry
- **Goals:** Minimize risks, promote fairness

**Speech:**
"AI governance refers to the framework designed to guide and oversee the development and application of AI technologies. It is essential for ensuring that AI is used ethically and responsibly. AI governance includes laws, regulations, and guidelines that direct both creators and users of AI. It involves multiple stakeholders, including governments, academia, and industry experts. The primary goals of AI governance are to minimize risks and promote fairness and transparency."

---

### Slide 6: Principles of AI Governance
- **Minimizing Risks:** Avoiding bias and harm
- **Fairness:** Ensuring equitable AI systems
- **Transparency:** Promoting understandable AI
- **Accountability:** Establishing responsibility
- **Public Trust:** Building societal confidence in AI

**Speech:**
"AI governance is built on several key principles. Minimizing risks involves avoiding bias and potential harm. Fairness ensures that AI systems are equitable and do not discriminate. Transparency promotes the development of understandable AI systems. Accountability establishes responsibility for AI outcomes, and public trust is crucial for building societal confidence in AI technologies."

---

### Slide 7: Tools for AI Governance
- **Bias Detection:** IBM AI Fairness 360, Aequitas
- **Explainability:** LIME, SHAP
- **Risk Management:** AI Risk Management Framework Navigator
- **Privacy:** OpenMined, TensorFlow Privacy
- **Compliance:** Adhering to regulations and standards

**Speech:**
"There are several tools available to support AI governance. For bias detection, tools like IBM AI Fairness 360 and Aequitas are used. Explainability tools like LIME and SHAP help in understanding AI decisions. Risk management tools like the AI Risk Management Framework Navigator assist in identifying and mitigating risks. Privacy tools like OpenMined and TensorFlow Privacy ensure data protection. Compliance with regulations and standards is also a critical aspect of AI governance."

---

### Slide 8: Introduction to AI Ethics
- **Definition:** Moral principles guiding AI use
- **Importance:** Prevents misuse and harm
- **Challenges:** Bias, discrimination, privacy breaches
- **Stakeholders:** Developers, users, policymakers
- **Goals:** Ethical AI development and deployment

**Speech:**
"AI ethics refers to the moral principles that guide the use of AI technologies. It is crucial for preventing misuse and harm. AI ethics addresses challenges such as bias, discrimination, and privacy breaches. It involves various stakeholders, including developers, users, and policymakers. The primary goal of AI ethics is to ensure that AI is developed and deployed ethically."

---

### Slide 9: Examples of AI Ethics Issues
- **Bias in AI Models:** Discrimination based on race or gender
- **Privacy Concerns:** Unauthorized data access
- **Unintended Consequences:** Harmful AI decisions
- **Accountability:** Responsibility for AI actions
- **Transparency:** Understanding AI decision-making

**Speech:**
"There are several examples of AI ethics issues. Bias in AI models can lead to discrimination based on race or gender. Privacy concerns arise from unauthorized data access. Unintended consequences can result from harmful AI decisions. Accountability involves determining who is responsible for AI actions. Transparency is essential for understanding how AI makes decisions."

---

### Slide 10: Conclusion and Future Directions
- **Summary:** Key points on AI chips, governance, and ethics
- **Future Trends:** Advancements in AI technology
- **Challenges:** Addressing ethical and governance issues
- **Opportunities:** Responsible AI development
- **Call to Action:** Engage in ethical AI practices

**Speech:**
"In conclusion, we have explored the critical aspects of AI chips, governance, and ethics. AI chips are revolutionizing various industries, while AI governance ensures responsible development and use. AI ethics addresses the moral principles guiding AI technologies. As we move forward, advancements in AI technology will continue to present new challenges and opportunities. It is essential to engage in ethical AI practices to ensure that AI benefits society as a whole."

---

Feel free to customize the content and add visuals to make the presentation more engaging.

In [ ]:
# retriever.invoke(" Making further continuing appropriations for fiscal year 2024")

[Document(metadata={'source': 'https://www.congress.gov/bill/118th-congress/house-bill/6363/text?s=1&r=1'}, page_content="Alert: Navigation Close GO GO  Tip     Search Only:  Tip      Tip      Tip     Any House Committees Agriculture (93rd-118th) Agriculture (93rd-118th) Appropriations (93rd-118th) Appropriations (93rd-118th) Armed Services (106th-118th) Armed Services (106th-118th) Budget (94th-118th) Budget (94th-118th) Education and the Workforce (118th) Education and the Workforce (118th) Energy and Commerce (107th-118th) Energy and Commerce (107th-118th) Ethics (112th-118th) Ethics (112th-118th) Financial Services (107th-118th) Financial Services (107th-118th) Foreign Affairs (110th-118th) Foreign Affairs (110th-118th) Homeland Security (109th-118th) Homeland Security (109th-118th) House Administration (106th-118th) House Administration (106th-118th) Intelligence (Permanent Select) (95th-118th) Intelligence (Permanent Select) (95th-118th) Judiciary (93rd-118th) Judiciary (93rd-118

In [ ]:
### Router

from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from pydantic import BaseModel, Field

# Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["vectorstore", "web_search"] = Field(
        ...,
        description="Given a user question choose to route it to web search or a vectorstore.",
    )


# LLM with function call
structured_llm_router = llm.with_structured_output(RouteQuery)

# Prompt
system = """You are an expert to generate a ppt to expose important information about GPUs"""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{inquery}"),
    ]
)

question_router = route_prompt | structured_llm_router
question = "Generate a useful slide to make a great presentation about GPUs, do it extensive"
print(
    question_router.invoke(
        {"inquery": question }
    )
)

datasource='web_search'


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal

members = ["Researcher", "Coder"]
system_prompt = (
    "You are a supervisor tasked with managing a conversation between the"
    " following workers:  {members}. Given the following user request,"
    " respond with the worker to act next. Each worker will perform a"
    " task and respond with their results and status. When finished,"
    " respond with FINISH."
)
# Our team supervisor is an LLM node. It just picks the next agent to process
# and decides when the work is completed
options = ["FINISH"] + members

class routeResponse(BaseModel):
    next: Literal["FINISH", "Researcher", "Coder"]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        (
            "system",
            "Given the conversation above, who should act next?"
            " Or should we FINISH? Select one of: {options}",
        ),
    ]
).partial(options=str(options), members=", ".join(members))


def supervisor_agent(state):
    supervisor_chain = (
        prompt
        | llm.with_structured_output(routeResponse)
    )
    return supervisor_chain.invoke(state)

In [ ]:
### Retrieval Grader


# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# LLM with function call
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    It does not need to be a stringent test.Return a json."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader
# question = "agent memory"
docs = retriever.invoke(question)
doc_txt = docs[1].page_content
print(retrieval_grader.invoke({"question": question, "document": doc_txt}))

binary_score='no'


In [ ]:
### Generate

from langchain import hub
from langchain_core.output_parsers import StrOutputParser

# Prompt
prompt = hub.pull("rlm/rag-prompt")


# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"context": format_docs(docs), "question": question})
print(generation)

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:351: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


I don't know.


https://www.datacamp.com/blog/ai-chips

https://www.datacamp.com/blog/ai-governance

https://www.datacamp.com/blog/ai-ethics-introduction